### Fera formidável 4.9 - Senha de tamanho variável

Alunos: José Victor da Silva Izidório e Lucas Nascimento da Silva

#### Enunciado:
Objetivo: Resolver o problema da senha de forma que você não forneça a informação
do tamanho da senha para a função que gera a população. Considere que a senha pode
ser uma string de 1 até 30 caracteres.

Dica: A função objetivo terá que quantificar em sua métrica tanto se o candidato
acertou as letras quanto se acertou o tamanho da senha.

Dica 2: Você pode criar diferentes estratégias de mutação, não precisa ser apensa
uma! Quem sabe uma função de mutação pode alterar letras e a outra pode alterar o
tamanho da senha? Ver o exercício “Praticamente um X-man!”.

Dica 3: Observe que você terá que pensar um pouco sobre como fará o cruzamento no
caso de senhas de tamanhos diferentes. Quem sabe tenha que fazer alguma consideração
adicional sobre quais são os valores possíveis para o ponto de corte…

### Mudanças em relação ao problema da senha original

Todas mudanças feitas no código estão descritas abaixo e foram feitas tomando como base o notebook Descobrindo a senha do professor Dr. Daniel Roberto Cassar, da referência 1.

**Criação do candidato**

A alteração feita na função criação do candidato foi fazer ela gerar senhas com um valor aleatório que representa seu tamanho e está dentro de um intervalo definido.

**Função objetivo**

Para a função objetivo, adicionamos na métrica de distância um valor que corresponde a diferença de tamanho entre o candidato e a senha original. Sendo assim, quanto menor essa diferença, menor será a distância entre ambos. Entretanto, a distância entre caracteres continua válida.

**Cruzamento**

Adaptamos o código da função cruzamento para funcionar se os indivíduos pai e mãe fossem de tamanho diferente. Se o maior indivíduo for o pai adaptamos ele o deixando do mesmo tamanho da mãe, e vice versa, retirando os n últimos genes do pai. Dessa forma, garantimos que eles possam trocar seus genes. Entretanto, para não perdernos a informação extra do maior indivíduo nós adicionamos os genes extras do maior indivíduo no filho 2. 

Exemplo:

`pai = ["a", "b", "c", "d", "e"]`

`mãe = ["f", "g", "h"]`

O filho 1 sempre terá o tamanho do menor progenitor, que nesse caso será a mãe, e poderá assumir, por exemplo, os valores:

`filho1 = ["a", "g", "c"]`

Já o filho 2 terá o tamanho do maior progenitor, que nesse caso será o pai, e poderá assumir, por exemplo, os valores:

`filho2 = ["f", "b", "h", "d", "e"]`
 
**Mutação**

No contexto de algoritmos genéticos aplicados à evolução de senhas, é essencial permitir mutações que alterem o tamanho das cadeias de caracteres. Caso contrário, se a população não contiver indivíduos com o mesmo número de caracteres da senha-alvo, o algoritmo pode nunca convergir para a solução correta.
 
Para lidar com isso, implementamos uma mutação específica que modifica o tamanho da senha. Quando essa mutação ocorre, o indivíduo pode ganhar ou perder um gene, com 50% de chance para cada possibilidade. A única exceção é quando o indivíduo possui apenas um gene — nesse caso, não é permitido remover um gene, pois a existência de uma senha vazia não é válida no contexto do problema.
 
Além dessa mutação, também foram adicionadas a `mutacao_salto` e a `mutacao_simples` que altera o o valor associado a informação genética.

In [24]:
import random
from string import ascii_lowercase, ascii_uppercase, digits

from funcoes_4 import populacao_senha as cria_populacao
from funcoes_4 import funcao_objetivo_pop_senha as funcao_objetivo
from funcoes_4 import selecao_torneio_min as funcao_selecao
from funcoes_4 import cruzamento_uniforme as funcao_cruzamento
from funcoes_4 import mutacao_simples as funcao_mutacao1
from funcoes_4 import mutacao_salto as funcao_mutacao2
from funcoes_4 import mutacao_insecao_delecao as funcao_mutacao3

In [25]:
SENHA = list("UniaoSantozeiros2025")
CARACTERES_POSSIVEIS = ascii_lowercase + ascii_uppercase + digits

TAMANHO_POPULACAO = 100
CHANCE_DE_CRUZAMENTO = 0.5
CHANCE_DE_MUTACAO = 0.025
TAMANHO_TORNEIO = 3
TAMANHO_MINIMO_SENHA = 1
TAMANHO_MAXIMO_SENHA = 30

populacao = cria_populacao(TAMANHO_POPULACAO, TAMANHO_MINIMO_SENHA, TAMANHO_MAXIMO_SENHA, CARACTERES_POSSIVEIS)
menor_fitness_geral = float("inf")
geracao = 0

In [26]:
while menor_fitness_geral != 0:
    
    # Seleção
    fitness = funcao_objetivo(populacao, SENHA)        
    selecionados = funcao_selecao(populacao, fitness, TAMANHO_TORNEIO)
    
    # Cruzamento
    proxima_geracao = []
    for pai, mae in zip(selecionados[::2], selecionados[1::2]):
        individuo1, individuo2 = funcao_cruzamento(pai, mae, CHANCE_DE_CRUZAMENTO)
        proxima_geracao.append(individuo1)
        proxima_geracao.append(individuo2)
    
    # Mutação
    funcao_mutacao1(proxima_geracao, CHANCE_DE_MUTACAO, list(CARACTERES_POSSIVEIS))
    funcao_mutacao2(proxima_geracao, CHANCE_DE_MUTACAO, list(CARACTERES_POSSIVEIS))
    funcao_mutacao3(proxima_geracao, CHANCE_DE_MUTACAO, list(CARACTERES_POSSIVEIS))
    
    # Encerramento
    populacao = proxima_geracao
    geracao += 1
    
    fitness = funcao_objetivo(populacao, SENHA)
    menor_fitness_observado = min(fitness)
    
    if menor_fitness_observado < menor_fitness_geral:
        menor_fitness_geral = menor_fitness_observado
        indice = fitness.index(menor_fitness_observado)
        candidato = populacao[indice]
        print(geracao, "".join(candidato))

1 Vk
2 T
5 U
40 Un
106 Uni
139 Unia
246 Uniao
258 UniaoS
279 UniaoSa
316 UniaoSan
328 UniaoSant
344 UniaoSanto
363 UniaoSantoz
399 UniaoSantoze
426 UniaoSantozei
449 UniaoSantozeir
467 UniaoSantozeiro
499 UniaoSantozeiros
542 UniaoSantozeiros2
690 UniaoSantozeiros20
735 UniaoSantozeiros202
758 UniaoSantozeiros2025


### Conclusão

As alterações conseguiram resolver o problema. Algo que percebemos na forma como o algoritimo funciona foi que ele forma uma escada se aproximando da senha correta se olharmos para a senha que corresponde ao maior fit observado. Isso se deve ao fato de que nesse código é mais provável alcançar uma distância maior da senha com um número maior para a senha que tem chance maior de ter caracteres errados.

### Referência

1. Cassar, Daniel R. "ATP-303 GA 4.2 - Notebook descobrindo a senha".